# Module 4 — Landslide Risk Model

Multi-hazard climate risk system for Indian real estate.  
This notebook builds a landslide susceptibility model for 10 hilly/mountain cities using:
- **Open-Meteo Archive API** — daily rainfall (2015-2023)
- **OpenTopoData SRTM30m** — elevation
- **XGBoost classifier** — trained on engineered features


In [ ]:
# Cell 1 — Fetch rainfall data for all 10 cities

import requests
import pandas as pd
import numpy as np
import time
import os

CITIES = [
    {"name": "Shimla",     "lat": 31.10, "lon": 77.17},
    {"name": "Manali",     "lat": 32.24, "lon": 77.19},
    {"name": "Darjeeling", "lat": 27.04, "lon": 88.27},
    {"name": "Gangtok",    "lat": 27.34, "lon": 88.61},
    {"name": "Shillong",   "lat": 25.58, "lon": 91.89},
    {"name": "Dehradun",   "lat": 30.32, "lon": 78.03},
    {"name": "Mussoorie",  "lat": 30.46, "lon": 78.07},
    {"name": "Ooty",       "lat": 11.41, "lon": 76.70},
    {"name": "Munnar",     "lat": 10.09, "lon": 77.06},
    {"name": "Kozhikode",  "lat": 11.26, "lon": 75.78}
]

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

frames = []
for city in CITIES:
    print(f"Fetching rainfall for {city['name']}...")
    params = {
        "latitude": city["lat"],
        "longitude": city["lon"],
        "start_date": "2015-01-01",
        "end_date": "2023-12-31",
        "daily": "precipitation_sum",
        "timezone": "Asia/Kolkata"
    }
    resp = requests.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    df = pd.DataFrame({
        "date": pd.to_datetime(data["daily"]["time"]),
        "rainfall_mm": data["daily"]["precipitation_sum"],
        "city": city["name"],
        "lat": city["lat"],
        "lon": city["lon"]
    })
    frames.append(df)
    time.sleep(0.3)

df_all = pd.concat(frames, ignore_index=True)
df_all["rainfall_mm"] = df_all["rainfall_mm"].fillna(0)

print(f"\nTotal rows: {len(df_all)}")
print(df_all.head())
print(df_all.tail())

In [ ]:
# Cell 2 — Fetch elevation for all 10 cities (one batch request)

locations_str = "|".join([f"{c['lat']},{c['lon']}" for c in CITIES])
topo_url = f"https://api.opentopodata.org/v1/srtm30m?locations={locations_str}"

print("Fetching elevation data...")
resp_elev = requests.get(topo_url, timeout=30)
resp_elev.raise_for_status()
elev_data = resp_elev.json()

elevation_dict = {}
for city, result in zip(CITIES, elev_data["results"]):
    elevation_dict[city["name"]] = result["elevation"]

print("\nElevation dict:")
for k, v in elevation_dict.items():
    print(f"  {k}: {v} m")

In [ ]:
# Cell 3 — Feature engineering (yearly aggregation)

df_all["year"] = df_all["date"].dt.year
df_all["elevation_m"] = df_all["city"].map(elevation_dict)

# Compute 7-day rolling rainfall sum per city
df_all = df_all.sort_values(["city", "date"]).reset_index(drop=True)
df_all["rolling_7day"] = (
    df_all.groupby("city")["rainfall_mm"]
    .transform(lambda x: x.rolling(7).sum())
)

yearly_agg = df_all.groupby(["city", "year"]).agg(
    elevation_m=("elevation_m", "first"),
    max_7day_rainfall_mm=("rolling_7day", "max"),
    days_above_100mm=("rainfall_mm", lambda x: (x > 100).sum()),
    lat=("lat", "first"),
    lon=("lon", "first")
).reset_index()

print(f"Yearly aggregated rows: {len(yearly_agg)}")
print(yearly_agg.head(10))

In [ ]:
# Cell 4 — Landslide label

yearly_agg["landslide_label"] = (
    (yearly_agg["elevation_m"] > 500) &
    (yearly_agg["max_7day_rainfall_mm"] > 300)
).astype(int)

print("Label distribution:")
print(yearly_agg["landslide_label"].value_counts())
print(f"\nPositive rate: {yearly_agg['landslide_label'].mean():.2%}")

In [ ]:
# Cell 5 — Train XGBoost model

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score
)
import matplotlib.pyplot as plt

FEATURES = ["elevation_m", "max_7day_rainfall_mm", "days_above_100mm"]
TARGET = "landslide_label"

train = yearly_agg[yearly_agg["year"] <= 2021]
test  = yearly_agg[yearly_agg["year"] >= 2022]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train: {len(train)} rows  |  Test: {len(test)} rows")
print(f"Train label dist: {dict(y_train.value_counts())}")
print(f"Test  label dist: {dict(y_test.value_counts())}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print(f"\nAccuracy:  {accuracy_score(y_test, y_pred):.4f}")
try:
    auc = roc_auc_score(y_test, y_proba[:, 1])
    print(f"AUC-ROC:   {auc:.4f}")
except ValueError as e:
    print(f"AUC-ROC:   N/A ({e})")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1:        {f1_score(y_test, y_pred, zero_division=0):.4f}")

# Feature importance plot
os.makedirs("../data/outputs", exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
importance = model.feature_importances_
sorted_idx = np.argsort(importance)
ax.barh(
    [FEATURES[i] for i in sorted_idx],
    importance[sorted_idx],
    color="#2196F3"
)
ax.set_xlabel("Feature Importance")
ax.set_title("Landslide Model — Feature Importance")
plt.tight_layout()
fig.savefig("../data/outputs/landslide_feature_importance.png", dpi=150)
plt.show()
print("Saved: ../data/outputs/landslide_feature_importance.png")

In [ ]:
# Cell 6 — Risk scores

# Predict on full dataset
all_proba = model.predict_proba(yearly_agg[FEATURES])
yearly_agg["landslide_prob"] = all_proba[:, 1]
yearly_agg["landslide_risk_score"] = (all_proba[:, 1] * 100).round(2)

def categorize_risk(score):
    if score <= 25:
        return "LOW"
    elif score <= 50:
        return "MEDIUM"
    elif score <= 75:
        return "HIGH"
    else:
        return "VERY HIGH"

yearly_agg["risk_category"] = yearly_agg["landslide_risk_score"].apply(categorize_risk)

risk_scores = yearly_agg[[
    "city", "year", "lat", "lon",
    "landslide_label", "landslide_prob",
    "landslide_risk_score", "risk_category"
]].copy()

print("Risk category distribution:")
print(risk_scores["risk_category"].value_counts())
print(f"\n{risk_scores.head(15)}")

In [ ]:
# Cell 7 — Save outputs

import joblib

os.makedirs("../data/outputs", exist_ok=True)

risk_scores.to_csv("../data/outputs/landslide_risk_scores.csv", index=False)
print("Saved: ../data/outputs/landslide_risk_scores.csv")

joblib.dump(model, "../data/outputs/landslide_model.pkl")
print("Saved: ../data/outputs/landslide_model.pkl")

print(f"\nDone! {len(risk_scores)} risk scores generated for {risk_scores['city'].nunique()} cities.")